In [12]:
import math 
class Vector:
    def __init__(self,components):
        self.components = list(components)
        self.dim = len(self.components)
    def __add__(self,other):
        return Vector([a + b for a, b in zip(self.components, other.components)])
    def __sub__(self,other):
        return Vector([a - b for a, b in zip(self.components, other.components)])
    def dot(self,other):
        return sum(a*b for a,b in zip(self.components, other.components))
    def magnitude(self):
        return sum(x**2 for x in self.components) ** 0.5

    def normalize(self):
        mag = self.magnitude()
        return Vector([x / mag for x in self.components])

    def cosine_similarity(self, other):
        return self.dot(other) / (self.magnitude() * other.magnitude())

    def angle_between(self,other):
        return math.arccos(self.cosine_similarity(other)) * (180/math.pi)

    def __repr__(self):
        return f"Vector({self.components})"
    
                   

In [13]:
a = Vector([1,2,3])
b = Vector([4,5,6])

print(f"a + b = {a + b}")
print(f"a dot b = {a.dot(b)}")
print(f"|a| = {a.magnitude(): .4f}")
print(f" cosine similarity = {a.cosine_similarity(b): .4f}")

a + b = Vector([5, 7, 9])
a dot b = 32
|a| =  3.7417
 cosine similarity =  0.9746


In [14]:
class Matrix:
    def __init__(self, rows):
        self.rows = [list(row) for row in rows]
        self.shape = (len(self.rows),len(self.rows[0]))

    def __matmul__(self,other):
        if isinstance(other,Vector):
            return Vector([
                sum(self.rows[i][j] * other.components[j] for j in range(self.shape[1]))
                for i in range(self.shape[0])])
        rows = []
        for j in range(other.shape[1]):
            row = []
            row.append(sum(self.rows[i][k] * other.rows[k][j] for k in range(self.shape[1])))
            rows.append(row)
        return Matrix(rows)
    def transpose(self):
        return Matrix([[self.rows[j][i] for j in range(self.shape[0])]
                       for i in range(self.shape[1])])

    def __repr__(self):
        return f"Matrix({self.rows})"
    
        
        

In [15]:
rotation_90 = Matrix([[0,-1] ,[1,0]])
point = Vector([3,1])
rotated = rotation_90 @ point
print(f"Original: {point}")
print(f"Rotated 90 degrees: {rotated}")

Original: Vector([3, 1])
Rotated 90 degrees: Vector([-1, 3])


In [16]:
import random

random.seed(42)
weights = Matrix([[random.gauss(0, 0.1) for _ in range(3)] for _ in range(2)])
input_vector = Vector([1.0, 0.5, -0.3])

output = weights @ input_vector
print(f"Input (3D): {input_vector}")
print(f"Output (2D): {output}")
print("This is what a neural network layer does -- matrix multiplication.")

Input (3D): Vector([1.0, 0.5, -0.3])
Output (2D): Vector([-0.019714737127338927, 0.10873956075097067])
This is what a neural network layer does -- matrix multiplication.


***Julia Version***

In [1]:

using LinearAlgebra
a = [1.0, 2.0, 3.0]
b = [4.0, 5.0, 6.0]

println("a + b = ", a + b)
println("a · b = ", a ⋅ b)       # Julia supports unicode operators
println("|a| = ", √(a ⋅ a))
println("cosine = ", (a ⋅ b) / (√(a ⋅ a) * √(b ⋅ b)))

# Matrix-vector multiplication
W = [0.1 -0.2 0.3; 0.4 0.5 -0.1]
x = [1.0, 0.5, -0.3]
println("Wx = ", W * x)
println("This is a neural network layer.")

a + b = [5.0, 7.0, 9.0]
a · b = 32.0
|a| = 3.7416573867739413
cosine = 0.9746318461970762
Wx = [-0.09, 0.68]
This is a neural network layer.


In [2]:
def is_lineraly_independent(vectors):
    n = len(vectors)
    dim = len(vectors[0].components)
    mat = Matrix([v.components[:] for v in vectors])
    rows = [row[:] for row in mat.rows]
    rank = 0
    for col in range(dim):
        pivot = None
        for row in range(rank, len(rows)):
            if abs(rows[row][col]) > 1e-10:
                pivot = row
                break
        if pivot is None:
            continue 
        rows[rank], rows[pivot] = rows[pivot], rows[rank]
        scale = rows[rank][col]
        rows[rank] = [x / scale for x in rows[rank]]
        for row in range(len(rows)):
            if row != rank and abs(rows[row][col]) > 1e-10:
                factor = rows[row][col]
                rows[row] = [rows[row][j] - factor * rows[rank][j] for j in range(dim)]
        rank+=1
    return rank == n


In [3]:
def project(a,b):
    scalar = a.dot(b) / b.dot(b)
    return Vector([scalar * x for x in b.components])

def gram_schmidt(vectors):
    orthonormal = []
    for v in vectors:
        w = v
        for u in orthonormal:
            proj = project(w,u)
            w = w - proj
        if w.magnitude() <1e-10:
            continue
        orthonormal.append(w.normalize())
    return orthonormal


In [13]:
v1 = Vector([1, 0, 0])
v2 = Vector([1, 1, 0])
v3 = Vector([1, 1, 1])
basis = gram_schmidt([v1, v2, v3])
for i, u in enumerate(basis):
    print(f"u{i+1} = {u}")
    print(f"  |u{i+1}| = {u.magnitude():.6f}")

print(f"u1 · u2 = {basis[0].dot(basis[1]):.6f}")
print(f"u1 · u3 = {basis[0].dot(basis[2]):.6f}")
print(f"u2 · u3 = {basis[1].dot(basis[2]):.6f}")

u1 = Vector([1.0, 0.0, 0.0])
  |u1| = 1.000000
u2 = Vector([0.0, 1.0, 0.0])
  |u2| = 1.000000
u3 = Vector([0.0, 0.0, 1.0])
  |u3| = 1.000000
u1 · u2 = 0.000000
u1 · u3 = 0.000000
u2 · u3 = 0.000000


In [4]:
import numpy as np 

a = np.array([1,2,3] , dtype= float)
b = np.array([4,5,6] , dtype = float)

print(f"a + b = {a + b}")
print(f"a · b = {np.dot(a,b)}")
print(f"|a| = {np.linalg.norm(a):.4f}")
print(f"cosine = {np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)):.4f}")
W = np.random.randn(2,3) * 0.1
x = np.array([1.0, 0.5 , -0.3])
print(f"Wx = {W @ x}")



a + b = [5. 7. 9.]
a · b = 32.0
|a| = 3.7417
cosine = 0.9746
Wx = [ 0.09772478 -0.0263462 ]


In [5]:
A = np.array([[1,2] ,[2,4]])
def is_linearly_independent_with_numpy(matrix):
    return np.linalg.matrix_rank(matrix) == len(matrix)
print(f"Rank: {np.linalg.matrix_rank(A)}")
print(f"True or False... If it's lineraly independent or not: {is_linearly_independent_with_numpy(A)}")


Rank: 1
True or False... If it's lineraly independent or not: False


In [6]:
a = np.array([3,4])
b = np.array([1,0])
proj = (np.dot(a,b) / np.dot(b,b)) * b
print(f"Projection of {a} onto {b} : {proj}")

Q, R = np.linalg.qr(np.random.randn(3, 3))
print(f"Q is orthogonal: {np.allclose(Q @ Q.T, np.eye(3))}")
print(f"R is upper triangular: {np.allclose(R, np.triu(R))}")


Projection of [3 4] onto [1 0] : [3. 0.]
Q is orthogonal: True
R is upper triangular: True


In [7]:
import torch 

x = torch.randn(3, requires_grad = True)
y = torch.tensor([1.0,0.0,0.1])
print(x,y)
similarity = torch.dot(x,y)
similarity.backward()

print(f"x = {x.data}")
print(f"y = {y.data}")
print(f"dot product = {similarity.item(): .4f}")
print(f"d(dot)/dx = {x.grad}")

tensor([ 0.0454, -0.4903, -1.4483], requires_grad=True) tensor([1.0000, 0.0000, 0.1000])
x = tensor([ 0.0454, -0.4903, -1.4483])
y = tensor([1.0000, 0.0000, 0.1000])
dot product = -0.0995
d(dot)/dx = tensor([1.0000, 0.0000, 0.1000])


Excersize #2: Creating a 2d Scaling matrix that doubles the x-coordinate and triples the y-coordinate then apply it to the vector [1,1]

In [8]:
def create_2d_scaling_matrix(sx, sy):
    return np.array([[sx, 0],
                     [0, sy]])

input_vector = np.array([1, 1])
scaling_matrix = create_2d_scaling_matrix(2, 3)
result = scaling_matrix @ input_vector
print(result)

[2 3]


Exersize#3: Given 5 random word-like vectors (dimension 50), find the two most similar using cosine similarity

In [9]:
rng = np.random.default_rng()
num_words = 5
vector_dim = 50
word_vectors = rng.standard_normal((num_words, vector_dim))

max_sim = -1
best_pair = (0, 0)

for i in range(num_words):
    for j in range(i + 1, num_words):
        cur_sim = np.dot(word_vectors[i], word_vectors[j]) / (np.linalg.norm(word_vectors[i]) * np.linalg.norm(word_vectors[j]))
        if cur_sim > max_sim:
            max_sim = cur_sim
            best_pair = (i, j)

print(f"Most similar pair: vectors {best_pair[0]} and {best_pair[1]}")
print(f"Similarity: {max_sim:.4f}")

Most similar pair: vectors 1 and 4
Similarity: 0.1120


Exercize#4: Verify that the Gram-Schmidt output is truly orthnormal: check that every pair has dot product 0 and every vector has magnitude 1

In [17]:
v1 = Vector([1,0,0])
v2 = Vector([1,1,0])
v3 = Vector([1,1,1])
basis = gram_schmidt([v1,v2,v3])

for i in range(len(basis)):
    for j in range(i + 1,len(basis)):
        cur_dot = basis[i].dot(basis[j])
        print(f"basis[{i}] · basis[{j}] = {cur_dot:.10f}")
for i in range(len(basis)):
    mag= basis[i].magnitude()
    print(f"|basis{i}]|= {mag:.10f}")

    
    

basis[0] · basis[1] = 0.0000000000
basis[0] · basis[2] = 0.0000000000
basis[1] · basis[2] = 0.0000000000
|basis0]|= 1.0000000000
|basis1]|= 1.0000000000
|basis2]|= 1.0000000000


Exercise#5: Create a 3*3 matrix with rank. Verify using the `rank()` method. Then explain what geometric object the columns span 

In [19]:
matrix = np.array([[1,2,3],
                   [3,6,9],
                    [10,11,13]])
print(f"Rank: {np.linalg.matrix_rank(matrix)}")

Rank: 2


Exercise#6: Project the Vector [1,2,3] onto [1,1,1]. What does the result represetn geometrically? 

In [23]:
a = np.array([1,2,3])
b = np.array([1,1,1])
proj = (np.dot(a,b) / np.dot(b,b)) * b
print(f"Projection of {a} onto {b} : {proj}")

Projection of [1 2 3] onto [1 1 1] : [2. 2. 2.]


```
